# ERP Paired — 01: Compute Evokeds

Computes per-subject condition averages, contrasts, and group grand averages
for a 2-condition (paired) ERP analysis.

**Requires:** final epoch files from the preprocessing pipeline (`*_final-epo.fif`)

**Output:** `<subject>_<window>_ave.fif` per subject, `grand_average_<window>_ave.fif` group level

In [ ]:
%load_ext autoreload
%autoreload 2

from eeg_toolkit import (
    load_config,
    find_subjects,
    compute_evokeds_subject,
    compute_evokeds_all,
    compute_grand_averages,
)

# ── Update these paths to point to your configs ──
cfg     = load_config('../../../configs/your_experiment.yaml')
cfg_erp = load_config('../../../configs/your_erp_analysis.yaml')

subjects = find_subjects(cfg)
print(f"Subjects: {len(subjects)}")
print(f"Windows:    {[w.name for w in cfg_erp.erp.windows]}")
print(f"Conditions: {list(vars(cfg_erp.conditions).keys())}")

In [ ]:
# ── Test with the first subject ──
test_subject = subjects[0]
print(f"Computing evokeds for: {test_subject}\n")

ok, trial_counts = compute_evokeds_subject(
    cfg, cfg_erp, test_subject,
    overwrite=True, verbose=True,
)

print(f"\nResult: {ok}")
print(f"Trial counts: {trial_counts}")

In [ ]:
# ── Compute evokeds for all subjects ──
evoked_summary = compute_evokeds_all(cfg, cfg_erp, overwrite=False, verbose=True)

In [ ]:
# ── Compute group grand averages ──
compute_grand_averages(cfg, cfg_erp, overwrite=True, verbose=True)

In [ ]:
# ── Verify grand averages ──
import mne
from eeg_toolkit.evoked import get_grand_average_path

for window in cfg_erp.erp.windows:
    print(f"\n=== {window.name} ===")
    ga = mne.read_evokeds(get_grand_average_path(cfg, window.name), verbose='WARNING')
    for evo in ga:
        print(f"  {evo.comment:30s} | nave={evo.nave:3d} | "
              f"{len(evo.ch_names)} ch | {evo.times[0]:.2f}s to {evo.times[-1]:.2f}s")